In [ ]:
!git clone https://github.com/Fardays90/underwater-image-enhancements.git

images_path = '/content/underwater-image-enhancements/inputs/'
outputs_path = '/content/underwater-image-enhancements/outputs/'

In [ ]:
def HE(img):
  yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)

  Y_channel = yuv[:, :, 0]
  Y_equalized = cv2.equalizeHist(Y_channel)

  yuv[:, :, 0] = Y_equalized

  resultImg = cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)
  return resultImg

In [ ]:
def CLAHE(img):
  LAB_img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
  L_channel = LAB_img[:, :, 0]

  clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))

  L_enhanced = clahe.apply(L_channel)

  LAB_img[:, :, 0] = L_enhanced

  result_img = cv2.cvtColor(LAB_img, cv2.COLOR_LAB2RGB)

  return result_img

In [ ]:
%cd /content
!pip install torch torchvision
!pip install opencv-python matplotlib numpy Pillow

In [ ]:
! [ ! -d "FUnIE-GAN" ] && git clone https://github.com/xahidbuffon/FUnIE-GAN.git
!mkdir -p models
! [ -f /content/FUnIE-GAN/PyTorch/nets/funiegan.py ] && mv /content/FUnIE-GAN/PyTorch/nets/funiegan.py /content/models/
! [ -f /content/FUnIE-GAN/PyTorch/models/funie_generator.pth ] && mv /content/FUnIE-GAN/PyTorch/models/funie_generator.pth /content/models/


In [ ]:
import sys
import torch
sys.path.insert(0, '/content/models')
from funiegan import GeneratorFunieGAN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GeneratorFunieGAN()
model.load_state_dict(torch.load('/content/models/funie_generator.pth'))
model.to(device)
model.eval()
print("Model loaded correctly")


In [ ]:
from torchvision import transforms
from PIL import Image
import numpy as np

def funiegan(img):
  img = Image.fromarray(img)
  transform = transforms.Compose([
      transforms.Resize((256, 256)),
      transforms.ToTensor(),
      transforms.Normalize([0.5, 0.5, 0.5], [0.5,0.5,0.5])
  ])
  x = transform(img).unsqueeze(0).to(device)
  with torch.no_grad():
    output = model(x)
  output = output.squeeze(0).permute(1,2,0).cpu().numpy()
  output = ((output * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)
  return output


In [ ]:
import matplotlib.pyplot as plt

def visualize_comparison(img, he_img, clahe_img, gan_img, t_he, t_clahe, t_gan, visualize=False, save_path=None):
  fig, axes = plt.subplots(1, 4, figsize=(16, 5))

  axes[0].imshow(img)
  axes[0].set_title("Original")
  axes[0].axis("off")

  axes[1].imshow(he_img)
  axes[1].set_title(f"HE image\n{t_he:.4f}s", pad=10)
  axes[1].axis("off")

  axes[2].imshow(clahe_img)
  axes[2].set_title(f"CLAHE image\n{t_clahe:.4f}s", pad=10)
  axes[2].axis("off")

  axes[3].imshow(gan_img)
  axes[3].set_title(f"FUnIE-GAN\n{t_gan:.4f}s", pad=10)
  axes[3].axis("off")

  plt.tight_layout()
  if save_path:
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
  if visualize:
    plt.show()
  plt.close()


In [ ]:
from pathlib import Path
import cv2
import time

image_dir = Path(images_path)
image_paths = list(image_dir.glob('*'))
output_dir = Path(outputs_path)
first = True
for path in image_paths:
    img = cv2.imread(str(path))
    if img is None:
      print(f"Failed to load image at path: {path}")
      continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    start = time.time()
    he_img = HE(img)
    t_he = time.time() - start

    start = time.time()
    clahe_img = CLAHE(img)
    t_clahe = time.time() - start

    torch.cuda.synchronize()
    start = time.time()
    gan_img = funiegan(img)
    torch.cuda.synchronize()
    t_gan = time.time() - start

    save_path = output_dir / f"{path.stem}_comparison.png"
    if first:
      visualize_comparison(img, he_img, clahe_img, gan_img, t_he, t_clahe, t_gan, True, save_path)
      first = False
      continue
    visualize_comparison(img, he_img, clahe_img, gan_img, t_he, t_clahe, t_gan, False, save_path)







